# QuantiPhy · IIIXR 2D Motion (`g_box`) · Sanity (4 câu / 3 video trên GPU T4 ×2)

Kiểm tra đường đi từ raw video + câu hỏi QuantiPhy validation tới đại lượng vật lý `fresh_box` trên 4 câu mốc:
- `#399`, `#401` (`simulation_0163`, dùng chung track video, kiểm tra gia tốc tức thời và so `#401` với `cached_point`)
- `#142` (`simulation_0089`, kiểm tra khác biệt tag `tv=v3nt` vs `tv=v2`)
- `#2234` (`internet_0019`, kiểm tra thành phần vận tốc ngang `hvel` + chuẩn hóa trọng trường `g = 9.8 m/s²`)

Đồng thời chạy phép thử đối chứng **bật/tắt trace** cùng `seed=0` trên `#399`, kiểm chứng tái tính `fresh_box` trên CPU từ mảng `.npy` đã lưu (`run/verify.py`), rồi đóng gói `/kaggle/working/iiixr_box_sanity.tar` kèm trang phân tích lỗi offline `run/index.html`.

**Thiết lập Kaggle:** Chọn Accelerator **GPU T4 ×2** và bật **Internet** (hoặc gắn snapshot dataset/model + `quantiphy_full.tar` qua **Add Input** khi chạy offline).

## 1. Cấu hình

In [ ]:
REPO_URL = "https://github.com/leetranwyng/quantiphy_trannl.git"
SOURCE_REF = "main"  # Có thể pin trực tiếp commit SHA cụ thể
GITHUB_TOKEN_SECRET = "GITHUB_TOKEN"  # Chỉ dùng nếu repo chuyển sang Private

# Để trống để tự tải snapshot đã pin từ HuggingFace; hoặc điền đường dẫn trong /kaggle/input khi chạy offline.
DATASET_DIR = ""
MODELS_DIR = ""
CAW_FULL_TAR = ""  # Để trống để tự tìm quantiphy_full.tar trong /kaggle/input nếu đã gắn
WHEELHOUSE = ""

DTYPE_PROFILE = "t4_fp16"  # "t4_fp16" (tối ưu T4 SM 7.5), "fp32", hoặc "upstream_bf16"
RULE_PARSE_ONLY = False     # False = chạy cả Qwen3-VL-8B cho bước parse fallback + vlm_ground_notext
OUTPUT_BUDGET_GIB = 18.0

## 2. Clone repo tại ref đã chọn & cài đặt dependencies đã khóa

In [ ]:
import base64, os, subprocess, sys, tempfile
from pathlib import Path

source = Path(tempfile.mkdtemp(prefix="iiixr-box-")) / "repo"
env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
clone = ["git", "-c", "credential.helper=", "clone", "--quiet", REPO_URL, str(source)]
if subprocess.run(clone, env=env).returncode:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret(GITHUB_TOKEN_SECRET)
    auth = base64.b64encode(("x-access-token:" + token).encode()).decode()
    subprocess.run(clone, check=True, env=dict(env, GIT_CONFIG_COUNT="1",
        GIT_CONFIG_KEY_0="http.https://github.com/.extraHeader",
        GIT_CONFIG_VALUE_0="Authorization: Basic " + auth))
    del token, auth

remote_ref = f"refs/remotes/origin/{SOURCE_REF}"
is_branch = subprocess.run(["git", "show-ref", "--verify", "--quiet", remote_ref], cwd=source).returncode == 0
subprocess.run(["git", "checkout", "--quiet", "--detach", remote_ref if is_branch else SOURCE_REF], cwd=source, check=True)
print("Resolved source commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=source, text=True).strip())

install = [sys.executable, "-m", "pip", "install", "-q", "-r", str(source / "solutions/iiixr/kaggle/requirements.txt")]
subprocess.run(install + (["--no-index", "--find-links", WHEELHOUSE] if WHEELHOUSE else []), check=True)


## 3. Thực thi Sanity & xuất archive + viewer offline

In [ ]:
import runpy
sys.path.insert(0, str(source))
launch = runpy.run_path(str(source / "solutions/iiixr/kaggle/notebook.py"))["launch"]
SUMMARY = launch(source, "sanity", globals())
